In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq()
model = "llama-3.3-70b-versatile"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.chat.completions.create(**params)
    return message.choices[0].message.content

In [38]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "solution criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,'```json')
    return chat(messages,stop_sequences=['```'])


In [39]:
dataset = json.loads(generate_dataset())
print(dataset)


[{'task': 'Extract the bucket name from an AWS S3 URL using a regular expression.', 'solution criteria': 'The regex should match the bucket name correctly, and it should be case-sensitive.'}, {'task': "Write a Python function to validate an AWS IAM policy document that checks if a policy has a version '2012-10-17'.", 'solution criteria': 'The function should return True for valid policies and False for invalid policies.'}, {'task': 'Create a JSON object that represents an AWS SQS queue configuration, including the queue name, delay seconds, and message retention period.', 'solution criteria': 'The JSON object should have the required keys (queue name, delay seconds, and message retention period) with valid values.'}]


In [40]:
with open('dataset.json','w') as f:
    json.dump(dataset,f,indent=2)

In [41]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [42]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case[task]}
    Solution: {output}
    Solution Criteria : {test_case[solution criteria]}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [43]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    model_grade = grade_by_model(test_case,output)
    score = model_grade['score']
    reasoning = model_grade['reasoning']
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning":reasoning
    }

In [44]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_result = mean([result["score"] for result in results])
    print("Average Score :",average_result)
    return results

In [45]:
results = run_eval(dataset)

Average Score : 2


In [46]:
print(json.dumps(results,indent=2))

[
  {
    "output": "### Solution\n\nTo extract the bucket name from an AWS S3 URL using a regular expression, we can use the following Python code:\n\n```python\nimport re\n\ndef extract_bucket_name(url):\n    \"\"\"\n    Extracts the bucket name from an AWS S3 URL.\n\n    Args:\n        url (str): The AWS S3 URL.\n\n    Returns:\n        str: The bucket name.\n    \"\"\"\n    pattern = r'^https?://([a-zA-Z0-9-]+)\\.s3\\.[a-zA-Z0-9\\-]+\\.amazonaws\\.com'\n    match = re.match(pattern, url)\n\n    if match:\n        return match.group(1)\n    else:\n        return None\n\n# Example usage\nurl = \"https://my-bucket.s3.us-east-1.amazonaws.com/path/to/object\"\nbucket_name = extract_bucket_name(url)\n\nif bucket_name:\n    print(f\"Bucket name: {bucket_name}\")\nelse:\n    print(\"Invalid URL or bucket name not found\")\n```\n\n### Explanation\n\n*   The regular expression pattern `^https?://([a-zA-Z0-9-]+)\\.s3\\.[a-zA-Z0-9\\-]+\\.amazonaws\\.com` is used to match the bucket name in the